In [1]:
import copy
from pprint import pprint

import numpy as np
import pandas as pd
from parse import parse
from pathlib import Path

In [2]:
from ase.data import chemical_symbols

# Create the dictionary using a dict comprehension
num_to_symbol = {i: symbol for i, symbol in enumerate(chemical_symbols) if i > 0}

# Example lookup
#print(num_to_symbol[6])   # Output: 'C'
#print(num_to_symbol[12])  # Output: 'Mg'

In [3]:
def parse_grrm_saddlepoint_irc_log(log_file_path: Path, num_atoms = 12) -> dict:
    """This function parses a single log file from a specific assumed set of input parameters in a .com file submitted to GRRM.

    See the `assemble_com()` function for the pattern of the .com file which is assumed to be the basis for the log file to be
    parsed by this function. 

    
    """
    log_parsing_results = {"initial_TS_energy": None,
                           "initial_TS_geometry": None,
                           "initial_TS_frequencies": None,
                           "initial_TS_single_neg_freq_confirmed": False,
                           "saddle_point_found": False,
                           "reopt_TS_energy": None,
                           "reopt_TS_geometry": None,
                           "reopt_TS_num_steps": None,
                           "reopt_TS_frequencies": None,
                           "reopt_TS_single_neg_freq_confirmed": False,
                           "1st_irc_EQ_energy": None,
                           "1st_irc_EQ_geometry": None,
                           "2nd_irc_EQ_energy": None,
                           "2nd_irc_EQ_geometry": None,
                          }
    
    itr_header_pattern = "# ITR. {iter_num}"
    consuming_coordinates = False
    start_consuming_coordinates_at = -1 
    stop_consuming_coordinates_at = -1 
    consuming_energy = False
    consume_energy_at = -1
    saddle_point_found = False
    optimized_structure = False
    consuming_freq = False
    reopt_TS_frequencies = []
    
    # With saddle point optimization AND IRC, we have three stages of the log:
    # Stage 1: re-optimization of candidate TS to true saddle point.
    # Stage 2: first EQ-finding by IRC.
    # Stage 3: second EQ-finding by IRC.
    
    stage = 1 
    
    with open(log_file_path, "r") as f: 
        for i, line in enumerate(f):
            if line.startswith("# ITR."):
                iter_parsed = parse(itr_header_pattern, line)
                iter_num = int(iter_parsed["iter_num"])
                if stage == 1 and iter_num == 0:
                    consuming_energy = True
                    consume_energy_at = i + num_atoms + 2
                    consuming_coordinates = True
                    start_consuming_coordinates_at = i+1
                    stop_consuming_coordinates_at = i+num_atoms
                    temp_coords = []
            elif line.startswith("Optimized structure"):
                optimized_structure = True
                consuming_energy = True
                consume_energy_at = i + num_atoms + 1
                consuming_coordinates = True
                start_consuming_coordinates_at = i+1
                stop_consuming_coordinates_at = i+num_atoms
                temp_coords = []
            elif line.startswith("1st-Order Saddle point was found"):
                saddle_point_found = True
                log_parsing_results["saddle_point_found"] = saddle_point_found
                print(f"Saddle point was found after {iter_num} iterations.")
            elif saddle_point_found and line.startswith("FREQFREQFREQ") and len(reopt_TS_frequencies) == 0:
                #print("Toggling frequency parsing on.")
                consuming_freq = True
            elif (consuming_freq) and (line.startswith("Freq.  :")):
                temp_freqs = [float(x) for x in line.rstrip().split()[-3:]]
                for x in temp_freqs:
                    reopt_TS_frequencies.append(x)
                if len(reopt_TS_frequencies) == 30:
                    #print("Toggling frequency parsing off.\n")
                    consuming_freq = False
                    log_parsing_results["reopt_TS_frequencies"] = copy.deepcopy(reopt_TS_frequencies)
                    if reopt_TS_frequencies[0] < 0.0 and all([x >= 0.0 for x in reopt_TS_frequencies[1:]]):
                        log_parsing_results["reopt_TS_single_neg_freq_confirmed"] = True
            # Adding sanity check on initial state -- does it have a single negative frequency?
            elif line.startswith("NORMAL MODE EIGENVALUE : N_MODE = 30") and stage == 1 and iter_num == 0:
                consuming_freq = True
                initial_TS_frequencies = []
            elif consuming_freq and stage == 1 and iter_num == 0:
                temp_freqs = [float(x) for x in line.strip().split()]
                for x in temp_freqs:
                    initial_TS_frequencies.append(x)
                if len(initial_TS_frequencies) == 30:
                    consuming_freq = False
                    log_parsing_results["initial_TS_frequencies"] = copy.deepcopy(initial_TS_frequencies)
                    if initial_TS_frequencies[0] < 0.0 and all([x >= 0.0 for x in initial_TS_frequencies[1:]]):
                        log_parsing_results["initial_TS_single_neg_freq_confirmed"] = True
            if consuming_coordinates and start_consuming_coordinates_at <= i:
                temp_coord_row = [line.strip().split()[0]] + [float(x) for x in line.rstrip().split()[1:]]
                temp_coords.append(temp_coord_row)
                if stop_consuming_coordinates_at == i:
                    consuming_coordinates = False
                    if stage == 1 and iter_num == 0:
                        log_parsing_results["initial_TS_geometry"] = copy.deepcopy(temp_coords)
                    elif optimized_structure:
                        if stage == 1:
                            log_parsing_results["reopt_TS_geometry"] = copy.deepcopy(temp_coords)
                        elif stage == 2:
                            log_parsing_results["1st_irc_EQ_geometry"] = copy.deepcopy(temp_coords)
                        elif stage == 3:
                            log_parsing_results["2nd_irc_EQ_geometry"] = copy.deepcopy(temp_coords)
            if consuming_energy and consume_energy_at == i:
                if stage == 1 and iter_num == 0:
                    log_parsing_results["initial_TS_energy"] = float(line.split()[1])
                elif optimized_structure:
                    print("Found optimized structure and energy.")
                    temp_energy = float(line.rstrip().split()[2])
                    if stage == 1:
                        log_parsing_results["reopt_TS_energy"] = temp_energy
                        log_parsing_results["reopt_TS_num_steps"] = iter_num
                    elif stage == 2:
                        log_parsing_results["1st_irc_EQ_energy"] = temp_energy
                    elif stage == 3:
                        log_parsing_results["2nd_irc_EQ_energy"] = temp_energy
                    optimized_structure = False
                    if stage <= 2:
                        stage += 1
                consuming_energy = False
                consume_energy_at = -1
    return log_parsing_results

In [4]:
# DONE add check if initial state has single negative frequency --> sanity check. 
# If so, it should not be very different eventual saddle-point. An IRC EQs should match. 

In [5]:
#pprint(log_parsing_results, sort_dicts=False)

In [6]:
def assemble_xyz(z: list, pos: np.array) -> str:
    """Assembling atomic numbers and positions into xyz format

    Args:
        z (list): chemical elements
        pos (tensor): 3D coordinates

    Returns:
        str: xyz string
    """
    natoms =len(z)
    xyz = f"{natoms}\n\n"
    for _z, _pos in zip(z, pos): #.numpy()):
        xyz += f"{_z}\t" + "\t".join([str(x) for x in _pos]) + "\n"
    return xyz

In [7]:
def assemble_com(geom: list) -> str:
    """Assembling atomic numbers and positions into .com format

    Args:
        z (list): chemical elements
        pos (tensor): 3D coordinates

    Returns:
        str: xyz string
    """
    coords_block = "\n".join(["\t".join([str(y) for y in x]) for x in geom])+"\n"
    return f'''# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
{coords_block.rstrip()}
Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200
'''

In [8]:
log_parsing_results = parse_grrm_saddlepoint_irc_log("scratch/C6H6_saddle_validation/C6H6_irc-val_ref_example.log")
# NB 'C6H6_5710-TS12 CON(2,6)' --> "scratch/C6H6_saddle_validation/C6H6_irc-val_ref_example.log"

com_block = assemble_com(log_parsing_results["initial_TS_geometry"])
print(com_block)

Found optimized structure and energy.
Saddle point was found after 2 iterations.
Found optimized structure and energy.
Found optimized structure and energy.
# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C	-0.03560272	-0.27134174	-1.5551164
C	-1.0147139	-0.04375157	-0.58207583
C	1.2424026	0.083484	-1.1180137
C	-0.54334307	0.46321496	0.7129997
C	1.2926998	0.3485111	0.27941152
C	-0.586081	-0.48620486	1.7599235
H	-2.0806084	-0.25336453	-0.7431103
H	1.7082402	1.3001873	0.6440455
H	-0.81180584	-1.5127023	1.3673085
H	1.745792	-0.49068445	0.8312485
H	-0.64512765	1.5350078	0.94996226
H	-0.2718522	-0.6723557	-2.5465832
Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200



In [9]:
#pprint(log_parsing_results, sort_dicts=False)

In [10]:
#with open("scratch/C6H6_saddle_validation/recreation-val-C6H6_irc-val_new_model_example.com", "w") as f:
#    f.write(com_block)

In [11]:
!cat scratch/C6H6_saddle_validation/C6H6_irc-val_new_model_example.com

# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C       -0.1385524172364524     -1.2895242342355837     -0.7607652901753402
C       -0.5746187159788264     1.1485610804884538      0.848903484647897
C       -0.43944556590990613    -1.3527369867314836     0.5051870850523872
C       0.23428055555705848     -0.05296881440221325    -1.4043258483621284
C       0.47971960191411184     0.8922688413213409      -0.054627277359575295
C       0.1317284754451082      -0.05383297277965539    1.1715038246435192
H       -1.6438144149385339     1.3403581628136956      0.7009595484863388
H       1.1587655352566293      -0.050997884501638606   -2.0300235898723886
H       -0.9498021996291618     -2.1129675287441425     1.1082410361243584
H       0.8810486914032343      -0.2468647014568118     1.942075259586514
H       1.441659302798145       1.397399397067585       -0.05456898430019256
H       -0.5809688486814069     0.3813056411604537      -1.9725592484713896
Options
Saddle+IRC
Do

In [12]:
!cat scratch/C6H6_saddle_validation/recreation-test-C6H6_irc-val_new_model_example.com

# SADDLE/B3LYP/Def2SVPP Int(Grid=FineGrid) EmpiricalDispersion=GD3

0 1
C	-0.138552417236	-1.289524234236	-0.760765290175
C	-0.574618715979	1.148561080488	0.848903484648
C	-0.43944556591	-1.352736986731	0.505187085052
C	0.234280555557	-0.052968814402	-1.404325848362
C	0.479719601914	0.892268841321	-0.05462727736
C	0.131728475445	-0.05383297278	1.171503824644
H	-1.643814414939	1.340358162814	0.700959548486
H	1.158765535257	-0.050997884502	-2.030023589872
H	-0.949802199629	-2.112967528744	1.108241036124
H	0.881048691403	-0.246864701457	1.942075259587
H	1.441659302798	1.397399397068	-0.0545689843
H	-0.580968848681	0.38130564116	-1.972559248471

Options
Saddle+IRC
DownDC = 99999
EigenCheck
GauProc=40
GauMem=200


In [13]:
import pymatgen

In [14]:
import os

from ase.io import read, write

generated_val_instances_dir = os.path.abspath("/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/")

In [15]:
from ase.io import read
import io

def read_xyz_with_raw_comments(filename):
    """
    Reads an XYZ trajectory and treats the second line of each frame 
    strictly as a single string comment.
    """
    with open(filename, 'r') as f:
        while True:
            # 1. Read the number of atoms
            line = f.readline()
            if not line: break
            natoms = int(line.strip())
            
            # 2. Grab the entire comment line as a single string
            raw_comment = f.readline().strip()
            
            # 3. Read the coordinate block
            coords_lines = [f.readline() for _ in range(natoms)]
            
            # 4. Use io.StringIO to let ASE parse only the coordinates
            xyz_data = f"{natoms}\n{raw_comment}\n" + "".join(coords_lines)
            atoms = read(io.StringIO(xyz_data), format='xyz')
            
            # 5. Force the comment into the info dict as a single value
            atoms.info["comment"] = raw_comment
            
            yield atoms

## 01/07/2026

Cleaned up a bit, made the log parsing loop into a per-log-file function. 

In [16]:
log_files_dir = Path("/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/log-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68/") # UPDATE

log_files = [f for f in os.listdir(log_files_dir) if os.path.isfile(os.path.join(log_files_dir, f))]
len(log_files)

100

### 1. Compare one log file's results to its respective reference data. 



In [17]:
log_file = log_files[6]
log_parsing_results = parse_grrm_saddlepoint_irc_log(log_files_dir / log_file)

parsed = parse("{network_name}-TS{ts_id_num}_CON_{eq_id_1}-{eq_id_2}_.log", log_file)
network_name = parsed['network_name'] 
ts_id_num  = parsed['ts_id_num']
eq_id_1 = parsed['eq_id_1']
eq_id_2 = parsed['eq_id_2']
ts_xyz_file = f"{network_name}-TS{ts_id_num} CON({eq_id_1},{eq_id_2}).xyz"
ref_atoms = read(Path(generated_val_instances_dir) / ts_xyz_file, index=":")[0:3]
len(ref_atoms)



Found optimized structure and energy.
Saddle point was found after 7 iterations.
Found optimized structure and energy.
Found optimized structure and energy.


3

In [18]:
#pprint(log_parsing_results, sort_dicts=False) # TS initial state confirmed.

In [19]:
import ase
import py3Dmol
from io import StringIO

def draw_in_3dmol(mol: str, fmt: str = "xyz") -> py3Dmol.view:
    """Draw the molecule

    Args:
        mol (str): str content of molecule.
        fmt (str, optional): format. Defaults to "xyz".

    Returns:
        py3Dmol.view: output viewer
    """
    viewer = py3Dmol.view(1024, 576)
    viewer.addModel(mol, fmt)
    viewer.setStyle({'stick': {}, "sphere": {"radius": 0.36}})
    viewer.zoomTo()
    return viewer

def atoms_to_xyz_str(atoms: ase.Atoms):
    f = StringIO()
    atoms.write(f, format="xyz")
    return f.getvalue()

def xyz_str_to_atoms(xyz_str: str):
    atoms = read(StringIO(xyz_str), format="xyz")
    return atoms
# draw_in_3dmol(atoms_to_xyz_str(atoms[-1]))

In [20]:
atoms = read('/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/C6H6_5710-TS18622 CON(195,71).xyz', format='xyz',index=":")

In [21]:
!head -n 42 "/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/C6H6_5710-TS18622 CON(195,71).xyz"

12
True/calculated reference reactant state.
C	-0.64518625	0.2447242	-1.683171
C	0.13141951	-0.79333	-1.7420098
C	0.7221887	0.42801055	-1.0730848
C	-0.8208481	-0.30277622	2.2158253
C	0.9219525	0.49097505	0.4189521
C	0.055722147	0.097144775	1.3265454
H	-1.617357	0.71064174	-1.8389342
H	0.2999874	-1.8419049	-1.9835051
H	-0.832695	-1.3374037	2.5891464
H	1.8824017	0.905396	0.75994915
H	-1.576706	0.3812367	2.6290455
H	1.4791207	1.0172857	-1.6187587
12
True/calculated reference transition state.
C	-0.03560272	-0.27134174	-1.5551164
C	-1.0147139	-0.04375157	-0.58207583
C	1.2424026	0.083484	-1.1180137
C	-0.54334307	0.46321496	0.7129997
C	1.2926998	0.3485111	0.27941152
C	-0.586081	-0.48620486	1.7599235
H	-2.0806084	-0.25336453	-0.7431103
H	1.7082402	1.3001873	0.6440455
H	-0.81180584	-1.5127023	1.3673085
H	1.745792	-0.49068445	0.8312485
H	-0.64512765	1.5350078	0.94996226
H	-0.2718522	-0.6723557	-2.5465832
12
True/calculated reference product state.
C	-0.054861784	-0.26756024	-1.5816629
C	-1.0227

In [22]:
node_features_ref_ts_xyz = '''12
True/calculated reference transition state.
C	-0.03560272	-0.27134174	-1.5551164
C	-1.0147139	-0.04375157	-0.58207583
C	1.2424026	0.083484	-1.1180137
C	-0.54334307	0.46321496	0.7129997
C	1.2926998	0.3485111	0.27941152
C	-0.586081	-0.48620486	1.7599235
H	-2.0806084	-0.25336453	-0.7431103
H	1.7082402	1.3001873	0.6440455
H	-0.81180584	-1.5127023	1.3673085
H	1.745792	-0.49068445	0.8312485
H	-0.64512765	1.5350078	0.94996226
H	-0.2718522	-0.6723557	-2.5465832'''
draw_in_3dmol(node_features_ref_ts_xyz)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [23]:
# Note that this "reference state" is via xyz_block_from_node_features ... but why?

In [24]:
import pickle

npz_path_val = "/scr/trond/SCAN/C6H6_5710-filtered-valid.pkl"
val_pkl = pickle.load(open(npz_path_val, "rb"))
for i in range(len(val_pkl["transition_state"]["rxn"])):
    #if np.allclose(val_pkl["transition_state"]["positions"][i][0], np.array([-0.03560272, -0.27134174, -1.5551164]), rtol=1e-05):
    #val_pkl["transition_state"]["rxn"][i] == 'C6H6_5710-TS12 CON(2,6)':
    if val_pkl["transition_state"]["rxn"][i] == "C6H6_5710-TS18622 CON(195,71)":
        print(i)
        print(val_pkl["transition_state"]["rxn"][i])
        pos = np.array(val_pkl['transition_state']['positions'][i])
        z_num = val_pkl['transition_state']['charges'][i]
        num2sym = {1: "H", 6: "C"}
        val_xyz = assemble_xyz([num2sym[num] for num in z_num], pos)
        print(val_xyz)
        

6580
C6H6_5710-TS18622 CON(195,71)
12

C	-0.37653042916999996	-0.27134173101	-1.3394274234330001
C	-1.355641565687	-0.043751569897	-0.3663868979839999
C	0.901474901004	0.08348399821200002	-0.902324771343
C	-0.8842707927	0.463214961394	0.9286886711349999
C	0.951772074556	0.348511105048	0.495100504034
C	-0.927008762207	-0.48620486696599996	1.975612450387
H	-2.421535957442	-0.253364541942	-0.527421280324
H	1.3673124024879997	1.3001872912159997	0.859734388587
H	-1.152733510099	-1.512702294698	1.582997489117
H	1.404864356449	-0.490684448184	1.046937436034
H	-0.986055344707	1.535007827151	1.165651209646
H	-0.6127799024759999	-0.6723557303459999	-2.330894255843



In [25]:
draw_in_3dmol(val_xyz)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [26]:
# Is it possible that the output reference TS is just rigidly transformed? 
from pymatgen.analysis.molecule_matcher import GeneticOrderMatcher
from pymatgen.core import Molecule

threshold = 0.05

node_features_ref_ts_atoms = read(StringIO(node_features_ref_ts_xyz), format="xyz")
nf_ref_ts_mol = Molecule(list(node_features_ref_ts_atoms.symbols), node_features_ref_ts_atoms.positions)
val_ref_ts_atoms = read(StringIO(val_xyz), format="xyz")
val_ref_ts_mol = Molecule(list(val_ref_ts_atoms.symbols), val_ref_ts_atoms.positions)

matcher = GeneticOrderMatcher(val_ref_ts_mol, threshold=threshold)

try:
    best_rmsd = None
    best_mirror_rmsd = None
    results = matcher.fit(nf_ref_ts_mol)
    if results:
        results.sort(key=lambda x: x[1])
        best_match, best_rmsd = results[0]
        
    mirror_nf_ref_ts_mol = Molecule(nf_ref_ts_mol.species, -1 * nf_ref_ts_mol.cart_coords)
    mirror_results = matcher.fit(mirror_nf_ref_ts_mol)
    if mirror_results:
        results.sort(key=lambda x: x[1])
        best_mirror_match, best_mirror_rmsd = results[0]
    if best_rmsd is not None and best_mirror_rmsd is not None:
        if best_rmsd > best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 1")
        elif best_rmsd < best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
        elif best_rmsd == best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (identical irrespective of mirroring)")
    elif best_rmsd is not None:
        print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 2")
    elif best_mirror_rmsd is not None:
        print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
    else:
        print("Acceptable RMSD was not found.")
except Exception as e:
    print(f"Could not calculate RMSD on first ref EQ: {e}")


Lowest Coordinate-based RMSD: 0.0000 Å. 2


In [27]:
# 07/07/2026 a generated instance vs the corresponding reference TS:
from pymatgen.analysis.molecule_matcher import GeneticOrderMatcher
from pymatgen.core import Molecule


#npz_path_test = "/scr/trond/SCAN/C6H6_5710-filtered-test.pkl"
#test_pkl = pickle.load(open(npz_path_test, "rb"))
#for i in range(len(test_pkl["transition_state"]["rxn"])):
#    if test_pkl["transition_state"]["rxn"][i] == "C6H6_5710-TS18506 CON(1516,1193)":
#        print(i)
#        print(test_pkl["transition_state"]["rxn"][i])
#        pos = np.array(test_pkl['transition_state']['positions'][i])
#        z_num = test_pkl['transition_state']['charges'][i]
#        num2sym = {1: "H", 6: "C"}
#        ref_ts_xyz = assemble_xyz([num2sym[num] for num in z_num], pos)
#        print(ref_ts_xyz)

ref_ts_xyz = """12
rxn_id: C6H6_5710-TS18506 CON(1516,1193)
C	-0.395371364449	-0.316673510267	-0.859714708889
C	-1.3133640256529997	0.500756965612	0.082339213069
C	0.900348766259	-0.243651118827	-0.048823807288
C	-1.213050496413	-0.915177852558	0.281313598153
C	-0.145288937084	-0.14768312696099997	1.09037423546
C	-0.592547856122	0.910428403141	1.875343929532
H	-2.107772564189	1.233053935082	-0.043566090248
H	1.543544105131	0.643204557089	-0.18758803455300002
H	0.210664375245	1.680824048252	1.919229166014
H	1.48767332075	-1.177729058783	-0.050105182027
H	-1.9167184146889997	-1.7245794629169997	0.46747677497599993
H	-0.549249438774	-0.44277377886	-1.938011574188
"""

threshold = 0.05*12#*16

cand_ts_xyz = """12
Generated/inpainted transition state. RMSD: 0.06393 <C3><85>. By model /misc/home/guest50/OAReactDiff/oa_reactdiff/trainer/checkpoint/OAReactDiff-SCAN/C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0/ddpm-epoch=2932-val-totloss=457.68.ckpt.
C       0.7041449740542529      -1.0852270751325463     -0.27177135849416056
C       1.0109074398877227      0.33275102684657626     0.1679284330425601
C       -1.6960602823473436     -0.002333770558145587   -0.5140100330668979
C       -0.13616217329992653    0.39550625728653915     1.149155617116568
C       -0.4710135361597421     1.0243379864768223      -0.13364406753145547
C       -0.7006962062221415     -0.680059959244912      0.22836235324380863
H       1.1989070870860599      -1.8548748335589451     0.3455510834866801
H       1.9440372283612495      0.8720924022176575      0.21535808899493883
H       0.7973197830179217      -1.3444959657544395     -1.345558996345475
H       -0.581448386110722      2.06107179741583        -0.5112852479325734
H       -0.2651526947306179     0.6998644942335042      2.1974636711168687
H       -1.8047832335367129     -0.418632360227941      -1.527549543630862
"""

cand_ts_atoms = read(StringIO(cand_ts_xyz), format="xyz")
cand_ts_mol = Molecule(list(cand_ts_atoms.symbols), cand_ts_atoms.positions)
ref_ts_atoms = read(StringIO(ref_ts_xyz), format="xyz")
ref_ts_mol = Molecule(list(ref_ts_atoms.symbols), ref_ts_atoms.positions)

matcher = GeneticOrderMatcher(ref_ts_mol, threshold=threshold)

try:
    best_rmsd = None
    best_mirror_rmsd = None
    results = matcher.fit(cand_ts_mol)
    if results:
        #print(results)
        results.sort(key=lambda x: x[1])
        best_match, best_rmsd = results[0]
        print(best_rmsd)
        
    mirror_cand_ts_mol = Molecule(cand_ts_mol.species, -1 * cand_ts_mol.cart_coords)
    mirror_results = matcher.fit(mirror_cand_ts_mol)
    if mirror_results:
        #print(mirror_results)
        mirror_results.sort(key=lambda x: x[1])
        best_mirror_match, best_mirror_rmsd = mirror_results[0]
        print(best_mirror_rmsd)
    if best_rmsd is not None and best_mirror_rmsd is not None:
        if best_rmsd < best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 1")
        elif best_rmsd > best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
        elif best_rmsd == best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (identical irrespective of mirroring)")
    elif best_rmsd is not None:
        print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 2")
    elif best_mirror_rmsd is not None:
        print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
    else:
        print("Acceptable RMSD was not found.")
except Exception as e:
    print(f"Could not calculate RMSD on first ref EQ: {e}")


0.41085664123703397
0.0639303616681337
Lowest Coordinate-based RMSD: 0.0639 Å (generated structure mirrored)


In [28]:
# RS: My code was: \\
# ```
# GeneticOrderMatcher(AseAtomsAdaptor.get_molecule(atoms_ref), threshold=0.5).fit(AseAtomsAdaptor.get_molecule(atoms_copy))
# ```
from pymatgen.io.ase import AseAtomsAdaptor

atoms_ref = xyz_str_to_atoms(ref_ts_xyz)
atoms_copy = xyz_str_to_atoms(cand_ts_xyz)

matcher = GeneticOrderMatcher(AseAtomsAdaptor.get_molecule(atoms_ref), threshold=0.5)

results = matcher.fit(AseAtomsAdaptor.get_molecule(atoms_copy))

mirror_mol = Molecule(list(atoms_copy.symbols), -1* atoms_copy.positions)

mirror_results = matcher.fit(mirror_mol)

if results:
    results.sort(key=lambda x: x[1])
    best_match, best_rmsd = results[0]
    print(best_rmsd)
    print(best_match)
    print("")
if mirror_results:
    mirror_results.sort(key=lambda x: x[1])
    best_mirror_match, best_mirror_rmsd = mirror_results[0]
    print("Mirrored:")
    print(best_mirror_rmsd)
    print(best_mirror_match)
    print("")

if best_rmsd is not None and best_mirror_rmsd is not None:
    if best_rmsd < best_mirror_rmsd:
        print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 1")
    elif best_rmsd > best_mirror_rmsd:
        print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
    elif best_rmsd == best_mirror_rmsd:
        print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (identical irrespective of mirroring)")
elif best_rmsd is not None:
    print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 2")
elif best_mirror_rmsd is not None:
    print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
else:
    print("Acceptable RMSD was not found.")

0.41085664123703397
Full Formula (H6 C6)
Reduced Formula: HC
Charge = 0, Spin Mult = 1
Sites (12)
0 C    -0.510676    -0.552639    -0.761428
1 C    -1.124490     0.730433     0.100131
2 C     0.869860    -1.230915    -0.183903
3 C    -1.292730    -0.725178     0.468105
4 C     0.173214    -0.573993     0.857403
5 C     0.034624     0.952764     1.051511
6 H    -1.883534     1.407258    -0.259914
7 H     0.887123     1.578681     0.719252
8 H    -0.279373     1.252524     2.066294
9 H     1.855530    -0.775685    -0.367225
10 H    -2.131136    -1.379472     0.745947
11 H    -0.689545    -0.683779    -1.847906

Mirrored:
0.0639303616681337
Full Formula (H6 C6)
Reduced Formula: HC
Charge = 0, Spin Mult = 1
Sites (12)
0 C    -0.384871    -0.297729    -0.818856
1 C    -1.322450     0.569812     0.245872
2 C     0.923295    -0.277293    -0.053098
3 C    -1.174618    -0.887880     0.325929
4 C    -0.054990    -0.253457     1.142838
5 C    -0.659053     0.890982     1.714282
6 H    -2.108757  

In [29]:
# Now check reactant against reactant
# Is it possible that the output reference TS is just rigidly transformed? 
from pymatgen.analysis.molecule_matcher import GeneticOrderMatcher
from pymatgen.core import Molecule

threshold = 0.05

node_features_ref_reactant_xyz = """12
True/calculated reference reactant state.
C	-0.64518625	0.2447242	-1.683171
C	0.13141951	-0.79333	-1.7420098
C	0.7221887	0.42801055	-1.0730848
C	-0.8208481	-0.30277622	2.2158253
C	0.9219525	0.49097505	0.4189521
C	0.055722147	0.097144775	1.3265454
H	-1.617357	0.71064174	-1.8389342
H	0.2999874	-1.8419049	-1.9835051
H	-0.832695	-1.3374037	2.5891464
H	1.8824017	0.905396	0.75994915
H	-1.576706	0.3812367	2.6290455
H	1.4791207	1.0172857	-1.6187587
"""

n = 6580
pos = np.array(val_pkl['reactant']['positions'][n])
z_num = val_pkl['reactant']['charges'][n]
val_reactant_xyz = assemble_xyz([num2sym[num] for num in z_num], pos)

node_features_ref_reactant_atoms = read(StringIO(node_features_ref_reactant_xyz), format="xyz")
nf_ref_reactant_mol = Molecule(list(node_features_ref_reactant_atoms.symbols), node_features_ref_reactant_atoms.positions)
val_ref_reactant_atoms = read(StringIO(val_reactant_xyz), format="xyz")
val_ref_reactant_mol = Molecule(list(val_ref_reactant_atoms.symbols), val_ref_reactant_atoms.positions)

matcher = GeneticOrderMatcher(val_ref_reactant_mol, threshold=threshold)

try:
    best_rmsd = None
    best_mirror_rmsd = None
    results = matcher.fit(nf_ref_reactant_mol)
    if results:
        results.sort(key=lambda x: x[1])
        best_match, best_rmsd = results[0]
        
    mirror_nf_ref_reactant_mol = Molecule(nf_ref_reactant_mol.species, -1 * nf_ref_reactant_mol.cart_coords)
    mirror_results = matcher.fit(mirror_nf_ref_reactant_mol)
    if mirror_results:
        results.sort(key=lambda x: x[1])
        best_mirror_match, best_mirror_rmsd = results[0]
    if best_rmsd is not None and best_mirror_rmsd is not None:
        if best_rmsd < best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 1")
        elif best_rmsd > best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
        elif best_rmsd == best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (identical irrespective of mirroring)")
    elif best_rmsd is not None:
        print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 2")
    elif best_mirror_rmsd is not None:
        print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
    else:
        print("Acceptable RMSD was not found.")
except Exception as e:
    print(f"Could not calculate RMSD on first ref EQ: {e}")


Lowest Coordinate-based RMSD: 0.0000 Å. 2


In [30]:
# Now check product against product
# Is it possible that the output reference TS is just rigidly transformed? 
from pymatgen.analysis.molecule_matcher import GeneticOrderMatcher
from pymatgen.core import Molecule

threshold = 0.05

node_features_ref_product_xyz = """12
True/calculated reference product state.
C	-0.054861784	-0.26756024	-1.5816629
C	-1.022776	-0.040450305	-0.63054657
C	1.2539868	0.07198121	-1.1194708
C	-0.47470316	0.443067	0.66556364
C	1.1410601	0.33135659	0.33218676
C	-0.4721144	-0.42633396	1.8297315
H	-2.1012967	-0.18592374	-0.7801474
H	1.6670759	1.2183394	0.72078544
H	-0.59778714	-1.4787741	1.4569228
H	1.6269631	-0.5374051	0.823437
H	-0.69313323	1.507296	0.87382054
H	-0.27241358	-0.6355928	-2.5906198"""

n = 6580
pos = np.array(val_pkl['product']['positions'][n])
z_num = val_pkl['product']['charges'][n]
val_product_xyz = assemble_xyz([num2sym[num] for num in z_num], pos)

node_features_ref_product_atoms = read(StringIO(node_features_ref_product_xyz), format="xyz")
nf_ref_product_mol = Molecule(list(node_features_ref_product_atoms.symbols), node_features_ref_product_atoms.positions)
val_ref_product_atoms = read(StringIO(val_product_xyz), format="xyz")
val_ref_product_mol = Molecule(list(val_ref_product_atoms.symbols), val_ref_product_atoms.positions)

matcher = GeneticOrderMatcher(val_ref_product_mol, threshold=threshold)

try:
    best_rmsd = None
    best_mirror_rmsd = None
    results = matcher.fit(nf_ref_product_mol)
    if results:
        results.sort(key=lambda x: x[1])
        best_match, best_rmsd = results[0]
        
    mirror_nf_ref_product_mol = Molecule(nf_ref_product_mol.species, -1 * nf_ref_product_mol.cart_coords)
    mirror_results = matcher.fit(mirror_nf_ref_product_mol)
    if mirror_results:
        results.sort(key=lambda x: x[1])
        best_mirror_match, best_mirror_rmsd = results[0]
    if best_rmsd is not None and best_mirror_rmsd is not None:
        if best_rmsd < best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 1")
        elif best_rmsd > best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
        elif best_rmsd == best_mirror_rmsd:
            print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (identical irrespective of mirroring)")
    elif best_rmsd is not None:
        print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å. 2")
    elif best_mirror_rmsd is not None:
        print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
    else:
        print("Acceptable RMSD was not found.")
except Exception as e:
    print(f"Could not calculate RMSD on first ref EQ: {e}")


Lowest Coordinate-based RMSD: 0.0000 Å. 2


In [31]:
import pickle
from pymatgen.analysis.molecule_matcher import GeneticOrderMatcher
from pymatgen.core import Molecule

log_parsing_results = parse_grrm_saddlepoint_irc_log("scratch/C6H6_saddle_validation/C6H6_irc-val_ref_example.log")
# NB: "C6H6_5710-TS18622 CON(195,71)" --> "scratch/C6H6_saddle_validation/C6H6_irc-val_ref_example.log"

npz_path_val = "/scr/trond/SCAN/C6H6_5710-filtered-valid.pkl"
val_pkl = pickle.load(open(npz_path_val, "rb"))
num2sym = {1: "H", 6: "C"}

n = 6580 # instance index
threshold = 0.05

val_n_eq1_pos = np.array(val_pkl['reactant']['positions'][n])
val_n_eq1_z_num = val_pkl['reactant']['charges'][n]
val_n_eq1_symbols = [num2sym[num] for num in val_n_eq1_z_num]
val_n_eq1_xyz = assemble_xyz(val_n_eq1_symbols, val_n_eq1_pos)
val_n_eq1_atoms = read(StringIO(val_n_eq1_xyz), format="xyz")
val_n_eq1_mol = Molecule(list(val_n_eq1_atoms.symbols), val_n_eq1_atoms.positions)

val_n_eq2_pos = np.array(val_pkl['product']['positions'][n])
val_n_eq2_z_num = val_pkl['product']['charges'][n]
val_n_eq2_symbols = [num2sym[num] for num in val_n_eq2_z_num]
val_n_eq2_xyz = assemble_xyz(val_n_eq2_symbols, val_n_eq2_pos)
val_n_eq2_atoms = read(StringIO(val_n_eq2_xyz), format="xyz")
val_n_eq2_mol = Molecule(list(val_n_eq2_atoms.symbols), val_n_eq2_atoms.positions)


# Checking for EQ matches:
if log_parsing_results['1st_irc_EQ_geometry'] is not None and log_parsing_results['2nd_irc_EQ_geometry'] is not None:
    irc_eq1_species = [x[0] for x in log_parsing_results['1st_irc_EQ_geometry']]
    irc_eq1_coords = [x[1:] for x in log_parsing_results['1st_irc_EQ_geometry']]
    irc_eq1_mol = Molecule(irc_eq1_species, irc_eq1_coords)
    irc_eq2_species = [x[0] for x in log_parsing_results['2nd_irc_EQ_geometry']]
    irc_eq2_coords = [x[1:] for x in log_parsing_results['2nd_irc_EQ_geometry']]
    irc_eq2_mol = Molecule(irc_eq2_species, irc_eq2_coords)

    # Alt 1: This works:
    #ref_eq1_mol = val_ref_reactant_mol #Molecule(val_n_eq1_symbols, val_n_eq1_pos)
    #ref_eq2_mol = val_ref_product_mol #Molecule(val_n_eq2_symbols, val_n_eq2_pos)

    # Alt 2: This works too:
    ref_eq1_mol = Molecule(val_n_eq1_symbols, val_n_eq1_pos)
    ref_eq2_mol = Molecule(val_n_eq2_symbols, val_n_eq2_pos)

    # Alt 3: This also works and used the un-transformed data.
    #ref_eq1_mol = val_n_eq1_mol
    #ref_eq2_mol = val_n_eq2_mol

    
    # 1. Initialize with the target (mol2)
    # The matcher now knows the reference structure it needs to map to
    ref_eq1_matcher = GeneticOrderMatcher(ref_eq1_mol, threshold=threshold)
    ref_eq2_matcher = GeneticOrderMatcher(ref_eq2_mol, threshold=threshold)
    
    # 2. Fit the input molecule (mol1) to the target (mol2)
    for i, irc_eq in enumerate([irc_eq1_mol, irc_eq2_mol]):
        for j, matcher in enumerate([ref_eq1_matcher, ref_eq2_matcher]):
            print(f"Checking IRC EQ number {i+1} against reference EQ {j+1}:")
            try:
                best_rmsd = None
                best_mirror_rmsd = None
                results = matcher.fit(irc_eq)
                if results:
                    results.sort(key=lambda x: x[1])
                    best_match, best_rmsd = results[0]
                    
                mirror_irc_eq = Molecule(irc_eq.species, -1 * irc_eq.cart_coords)
                mirror_results = matcher.fit(mirror_irc_eq)
                if mirror_results:
                    results.sort(key=lambda x: x[1])
                    best_mirror_match, best_mirror_rmsd = results[0]
                if best_rmsd is not None and best_mirror_rmsd is not None:
                    if best_rmsd < best_mirror_rmsd:
                        print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å")
                    elif best_rmsd > best_mirror_rmsd:
                        print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
                    elif best_rmsd == best_mirror_rmsd:
                        print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (identical irrespective of mirroring)")
                elif best_rmsd is not None:
                    print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å")
                elif best_mirror_rmsd is not None:
                    print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
                else:
                    print("Acceptable RMSD was not found.")
            except Exception as e:
                print(f"Could not calculate RMSD on first ref EQ: {e}")
            print("")
else:
    print("Missing EQs after IRC calculation.")

Found optimized structure and energy.
Saddle point was found after 2 iterations.
Found optimized structure and energy.
Found optimized structure and energy.
Checking IRC EQ number 1 against reference EQ 1:
Acceptable RMSD was not found.

Checking IRC EQ number 1 against reference EQ 2:
Lowest Coordinate-based RMSD: 0.0002 Å

Checking IRC EQ number 2 against reference EQ 1:
Lowest Coordinate-based RMSD: 0.0068 Å

Checking IRC EQ number 2 against reference EQ 2:
Acceptable RMSD was not found.



In [32]:
print(val_n_eq1_symbols)
print(list(val_n_eq1_atoms.symbols))
print(val_n_eq2_symbols)
print(list(val_n_eq2_atoms.symbols))
print(val_n_eq1_pos)
print(val_n_eq1_atoms.positions)
print(val_n_eq2_pos)
print(val_n_eq2_atoms.positions)

['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H']
['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H']
['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H']
['C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H']
[[-0.98611396  0.24472421 -1.46748204]
 [-0.20950822 -0.79333004 -1.52632077]
 [ 0.38126095  0.42801059 -0.85739594]
 [-1.16177584 -0.30277618  2.43151416]
 [ 0.58102476  0.49097507  0.63464103]
 [-0.28520559  0.0971448   1.54223428]
 [-1.95828477  0.71064175 -1.62324521]
 [-0.0409403  -1.8419049  -1.76781615]
 [-1.17362271 -1.33740366  2.80483541]
 [ 1.54147397  0.90539596  0.9756381 ]
 [-1.91763379  0.38123673  2.84473434]
 [ 1.13819296  1.01728565 -1.4030697 ]]
[[-0.98611396  0.24472421 -1.46748204]
 [-0.20950822 -0.79333004 -1.52632077]
 [ 0.38126095  0.42801059 -0.85739594]
 [-1.16177584 -0.30277618  2.43151416]
 [ 0.58102476  0.49097507  0.63464103]
 [-0.28520559  0.0971448   1.54223428]
 [-1.95828477  0.71064175 -1.62324521]
 [-0.0409403  

In [33]:
# Log file initial TS state
species = [str(x[0]) for x in log_parsing_results["initial_TS_geometry"]]
coords = [x[1:] for x in log_parsing_results["initial_TS_geometry"]]
draw_in_3dmol(assemble_xyz(species, coords))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [34]:
# val split pickle TS geometry
val_n_TS_pos = np.array(val_pkl['transition_state']['positions'][n])
val_n_TS_z_num = val_pkl['transition_state']['charges'][n]
val_n_TS_symbols = [num2sym[num] for num in val_n_TS_z_num]
draw_in_3dmol(assemble_xyz(val_n_TS_symbols, val_n_TS_pos))

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [35]:
draw_in_3dmol(assemble_xyz(val_n_eq1_symbols, val_n_eq1_pos)) # reference EQ: val instance 0 reactant state

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [36]:
draw_in_3dmol(assemble_xyz(irc_eq1_mol.species, irc_eq1_mol.cart_coords)) # IRC-based EQ 1 from saddle+IRC on reference TS: val instance 0 transition state

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [37]:
draw_in_3dmol(assemble_xyz(irc_eq2_mol.species, irc_eq2_mol.cart_coords)) # IRC-based EQ 2 from saddle+IRC on reference TS: val instance 0 transition state

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [38]:
draw_in_3dmol(assemble_xyz(val_n_eq2_symbols, val_n_eq2_pos)) # reference EQ: val instance 0 product state

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [39]:
# 1. Greatly increase the threshold so it doesn't reject valid matches.
# 10.0 Å is large enough to capture the TS -> EQ geometric difference.
matcher = GeneticOrderMatcher(ref_eq1_mol, threshold=0.4733)

# 2. fit() returns a list of tuples containing the matches and RMSDs
results = matcher.fit(irc_eq2_mol)

assert ref_eq1_mol.species == irc_eq2_mol.species

if results:
    # 3. The list is sorted from best (lowest RMSD) to worst. 
    results.sort(key=lambda x: x[1])
    # Grab the first item in the list.
    # 3. Now the first item is guaranteed to be the lowest RMSD
    best_match, best_rmsd = results[0]
    
    print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å")
    # Note: best_match contains the list of mapped indices.
else:
    print("No match found. Ensure the molecules have the exact same chemical composition.")

Lowest Coordinate-based RMSD: 0.0068 Å


In [40]:
irc_eq2_mol.species

[Element C,
 Element C,
 Element C,
 Element C,
 Element C,
 Element C,
 Element H,
 Element H,
 Element H,
 Element H,
 Element H,
 Element H]

### 2. Loop over all the log files in directory, compile stats on the results

In [41]:
# DONE also compare with -1* EQ state, take the min RMSD one.

In [42]:
!ls -haltr /misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/log-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch\=2932-val-totloss\=457_68/*.log | wc -l

100


In [43]:
# import pickle
# from pymatgen.analysis.molecule_matcher import GeneticOrderMatcher
# from pymatgen.core import Molecule

from ase.data import chemical_symbols

# Create the dictionary using a dict comprehension
num_to_symbol = {i: symbol for i, symbol in enumerate(chemical_symbols) if i > 0}

def compare_ref_vs_log_eqs(log_parsing_results: dict, ref_pkl: dict, ref_instance_index: int, threshold :float=0.05, num_to_symbol :dict=num_to_symbol) -> dict:
    eq_matching_results = {"matched_irc_eq1_to_ref_eq1": False, 
                "RMSD_of_matched_irc_eq1_to_ref_eq1": None,
                "matched_irc_eq1_to_ref_eq2": False, 
                "RMSD_of_matched_irc_eq1_to_ref_eq2": None,
                "matched_irc_eq2_to_ref_eq1": False,
                "RMSD_of_matched_irc_eq2_to_ref_eq1": None,
                "matched_irc_eq2_to_ref_eq2": False,
                "RMSD_of_matched_irc_eq2_to_ref_eq2": None,
                "matched_mirrored_irc_eq1_to_ref_eq1": False, 
                "RMSD_of_matched_mirrored_irc_eq1_to_ref_eq1": None,
                "matched_mirrored_irc_eq1_to_ref_eq2": False, 
                "RMSD_of_matched_mirrored_irc_eq1_to_ref_eq2": None, 
                "matched_mirrored_irc_eq2_to_ref_eq1": False, 
                "RMSD_of_matched_mirrored_irc_eq2_to_ref_eq1": None, 
                "matched_mirrored_irc_eq2_to_ref_eq2": False, 
                "RMSD_of_matched_mirrored_irc_eq2_to_ref_eq2": None, 
               }
    ref_eq1_pos = np.array(ref_pkl['reactant']['positions'][ref_instance_index])
    ref_eq1_z_num = ref_pkl['reactant']['charges'][ref_instance_index]
    ref_eq1_symbols = [num_to_symbol[num] for num in ref_eq1_z_num]
    ref_eq1_xyz = assemble_xyz(ref_eq1_symbols, ref_eq1_pos)
    ref_eq1_atoms = read(StringIO(ref_eq1_xyz), format="xyz")
    ref_eq1_mol = Molecule(list(ref_eq1_atoms.symbols), ref_eq1_atoms.positions)
    
    ref_eq2_pos = np.array(ref_pkl['product']['positions'][ref_instance_index])
    ref_eq2_z_num = ref_pkl['product']['charges'][ref_instance_index]
    ref_eq2_symbols = [num_to_symbol[num] for num in ref_eq2_z_num]
    ref_eq2_xyz = assemble_xyz(ref_eq2_symbols, ref_eq2_pos)
    ref_eq2_atoms = read(StringIO(ref_eq2_xyz), format="xyz")
    ref_eq2_mol = Molecule(list(ref_eq2_atoms.symbols), ref_eq2_atoms.positions)
    
    # Checking for EQ matches:
    if log_parsing_results['1st_irc_EQ_geometry'] is not None and log_parsing_results['2nd_irc_EQ_geometry'] is not None:
        irc_eq1_species = [x[0] for x in log_parsing_results['1st_irc_EQ_geometry']]
        irc_eq1_coords = [x[1:] for x in log_parsing_results['1st_irc_EQ_geometry']]
        irc_eq1_mol = Molecule(irc_eq1_species, irc_eq1_coords)
        irc_eq2_species = [x[0] for x in log_parsing_results['2nd_irc_EQ_geometry']]
        irc_eq2_coords = [x[1:] for x in log_parsing_results['2nd_irc_EQ_geometry']]
        irc_eq2_mol = Molecule(irc_eq2_species, irc_eq2_coords)
    
        #ref_eq1_mol = Molecule(ref_eq1_symbols, ref_eq1_pos)
        #ref_eq2_mol = Molecule(ref_eq2_symbols, ref_eq2_pos)
        
        # 1. Initialize matchers with the reference states:
        ref_eq1_matcher = GeneticOrderMatcher(ref_eq1_mol, threshold=threshold)
        ref_eq2_matcher = GeneticOrderMatcher(ref_eq2_mol, threshold=threshold)
        
        for i, irc_eq in enumerate([irc_eq1_mol, irc_eq2_mol]):
            for j, matcher in enumerate([ref_eq1_matcher, ref_eq2_matcher]):
                #print(f"Checking IRC EQ number {i+1} against reference EQ {j+1}:")
                try:
                    best_rmsd = None
                    best_mirror_rmsd = None
                    results = matcher.fit(irc_eq)
                    if results:
                        results.sort(key=lambda x: x[1])
                        best_match, best_rmsd = results[0]
                    mirror_irc_eq = Molecule(irc_eq.species, -1 * irc_eq.cart_coords)
                    mirror_results = matcher.fit(mirror_irc_eq)
                    if mirror_results:
                        mirror_results.sort(key=lambda x: x[1])
                        best_mirror_match, best_mirror_rmsd = mirror_results[0]
                    
                    if best_rmsd is not None and best_mirror_rmsd is not None:
                        if best_rmsd < best_mirror_rmsd:
                            #print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å")
                            eq_matching_results[f"matched_irc_eq{i+1}_to_ref_eq{j+1}"] = True
                            eq_matching_results[f"RMSD_of_matched_irc_eq{i+1}_to_ref_eq{j+1}"] = best_rmsd
                        elif best_rmsd > best_mirror_rmsd:
                            #print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
                            eq_matching_results[f"matched_mirrored_irc_eq{i+1}_to_ref_eq{j+1}"] = True
                            eq_matching_results[f"RMSD_of_matched_mirrored_irc_eq{i+1}_to_ref_eq{j+1}"] = best_mirror_rmsd
                        elif best_rmsd == best_mirror_rmsd:
                            #print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (identical irrespective of mirroring)")
                            eq_matching_results[f"matched_irc_eq{i+1}_to_ref_eq{j+1}"] = True
                            eq_matching_results[f"RMSD_of_matched_irc_eq{i+1}_to_ref_eq{j+1}"] = best_rmsd
                            eq_matching_results[f"matched_mirrored_irc_eq{i+1}_to_ref_eq{j+1}"] = True
                            eq_matching_results[f"RMSD_of_matched_mirrored_irc_eq{i+1}_to_ref_eq{j+1}"] = best_mirror_rmsd
                    elif best_rmsd is not None:
                        #print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å")
                        eq_matching_results[f"matched_irc_eq{i+1}_to_ref_eq{j+1}"] = True
                        eq_matching_results[f"RMSD_of_matched_irc_eq{i+1}_to_ref_eq{j+1}"] = best_rmsd
                    elif best_mirror_rmsd is not None:
                        #print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
                        eq_matching_results[f"matched_mirrored_irc_eq{i+1}_to_ref_eq{j+1}"] = True
                        eq_matching_results[f"RMSD_of_matched_mirrored_irc_eq{i+1}_to_ref_eq{j+1}"] = best_mirror_rmsd
                    else:
                        print(f"Reference instance {ref_instance_index}: Acceptable RMSD was not found.")
                except Exception as e:
                    print(f"Exception thrown for reference instance {ref_instance_index}: {e}")
               #print("")
    else:
        print(f"Reference instance {ref_instance_index}: Missing EQs after IRC calculation.")
    return eq_matching_results

In [44]:
def compare_ref_vs_log_tss(log_parsing_results: dict, ref_pkl: dict, ref_instance_index: int, threshold :float=0.05, num_to_symbol :dict=num_to_symbol, mode :str="reopt") -> dict:
    ts_matching_results = {f"matched_{mode}_TS_to_ref_TS": False, 
                           f"RMSD_of_matched_{mode}_TS_to_ref_TS": None,
                           f"matched_mirrored_{mode}_TS_to_ref_TS": False, 
                           f"RMSD_of_matched_mirrored_{mode}_TS_to_ref_TS": None,
                          }
    ref_ts_pos = np.array(ref_pkl['transition_state']['positions'][ref_instance_index])
    ref_ts_z_num = ref_pkl['transition_state']['charges'][ref_instance_index]
    ref_ts_symbols = [num_to_symbol[num] for num in ref_ts_z_num]
    ref_ts_xyz = assemble_xyz(ref_ts_symbols, ref_ts_pos)
    ref_ts_atoms = read(StringIO(ref_ts_xyz), format="xyz")
    ref_ts_mol = Molecule(list(ref_ts_atoms.symbols), ref_ts_atoms.positions)

    if log_parsing_results["saddle_point_found"]:
        cand_ts_species = [x[0] for x in log_parsing_results[f'{mode}_TS_geometry']]
        cand_ts_coords = [x[1:] for x in log_parsing_results[f'{mode}_TS_geometry']]
        cand_ts_mol = Molecule(cand_ts_species, cand_ts_coords)
    else:
        print(f"Reference instance {ref_instance_index}: no saddle-point was found.")
        return ts_matching_results
    
    matcher = GeneticOrderMatcher(ref_ts_mol, threshold=threshold)
    
    try:
        best_rmsd = None
        best_mirror_rmsd = None
        results = matcher.fit(cand_ts_mol)
        if results:
            results.sort(key=lambda x: x[1])
            best_match, best_rmsd = results[0]
            
        mirror_cand_ts_mol = Molecule(cand_ts_mol.species, -1 * cand_ts_mol.cart_coords)
        mirror_results = matcher.fit(mirror_cand_ts_mol)
        if mirror_results:
            mirror_results.sort(key=lambda x: x[1])
            best_mirror_match, best_mirror_rmsd = mirror_results[0]
        if best_rmsd is not None and best_mirror_rmsd is not None:
            if best_rmsd < best_mirror_rmsd:
                ts_matching_results[f"matched_{mode}_TS_to_ref_TS"] = True
                ts_matching_results[f"RMSD_of_matched_{mode}_TS_to_ref_TS"] = best_rmsd
            elif best_rmsd > best_mirror_rmsd:
                #print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
                ts_matching_results[f"matched_mirrored_{mode}_TS_to_ref_TS"] = True
                ts_matching_results[f"RMSD_of_matched_mirrored_{mode}_TS_to_ref_TS"] = best_mirror_rmsd
            elif best_rmsd == best_mirror_rmsd:
                #print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (identical irrespective of mirroring)")
                ts_matching_results[f"matched_{mode}_TS_to_ref_TS"] = True
                ts_matching_results[f"RMSD_of_matched_{mode}_TS_to_ref_TS"] = best_rmsd
                ts_matching_results[f"matched_mirrored_{mode}_TS_to_ref_TS"] = True
                ts_matching_results[f"RMSD_of_matched_mirrored_{mode}_TS_to_ref_TS"] = best_mirror_rmsd
        elif best_rmsd is not None:
            #print(f"Lowest Coordinate-based RMSD: {best_rmsd:.4f} Å")
            ts_matching_results[f"matched_{mode}_TS_to_ref_TS"] = True
            ts_matching_results[f"RMSD_of_matched_{mode}_TS_to_ref_TS"] = best_rmsd
        elif best_mirror_rmsd is not None:
            #print(f"Lowest Coordinate-based RMSD: {best_mirror_rmsd:.4f} Å (generated structure mirrored)")
            ts_matching_results[f"matched_mirrored_{mode}_TS_to_ref_TS"] = True
            ts_matching_results[f"RMSD_of_matched_mirrored_{mode}_TS_to_ref_TS"] = best_mirror_rmsd
        else:
            print(f"Reference instance {ref_instance_index}: Acceptable RMSD was not found.")
    except Exception as e:
        print(f"Exception thrown for reference instance {ref_instance_index}: {e}")
    return ts_matching_results

In [45]:
# UPDATE
# NB remove esccape backslashes.
log_files_dir = Path("/misc/home/guest50/OAReactDiff/results/sample_random_TS-20260626-C6H6-filtered_new-w_wo-w_wo-w_wo-vs_www_valid/log-C6H6_5710-filtered-LBS_32x8-StepLR-cutoff_21-rep0-SCAN-leftnet8ff8f5a1c6d0_ddpm-epoch=2932-val-totloss=457_68/")
ref_pkl_path  = Path("/scr/trond/SCAN/C6H6_5710-filtered-test.pkl")

# RS: """
#     The most important measures for PI report are so far (from my opinion):
#     - % of initial TS has single negative eigenvalue (= imaginary frequency)
#     - % of successful TS re-optimization
#        - of these, % of both IRC EQs matched reference EQs
#     """
def batch_process_grrm_saddlepoint_irc_log_files(log_files_dir :Path, ref_pkl_path :Path, num_atoms :int=12, threshold :float=0.05, ts_cand_mode :str="initial") -> dict:
    batch_log_results = {}
    instance_results_pattern = {"initial_TS_has_single_negative_eigenvalue": None,
                                "TS_was_successfully_reoptimized": None,
                                "both_IRC_EQs_matched_ref_EQs": None,
                                "best_IRC_EQ_match_RMSDs": None,
                                "TS_reopt_Delta_energy": None,
                               }
    log_files = [f for f in os.listdir(log_files_dir) if os.path.isfile(log_files_dir/Path(f))]
    ref_pkl = pickle.load(open(ref_pkl_path, "rb"))
    for log_file in log_files:
        temp_instance_results = copy.deepcopy(instance_results_pattern)
        parsed = parse("{network_name}-TS{ts_id_num}_CON_{eq_id_1}-{eq_id_2}_.log", log_file) # C6H6_5710-TS15759_CON_255_99_.log
        try:
            network_name = parsed['network_name']
        except:
            print(log_file)
        ts_id_num = parsed['ts_id_num']
        eq_id_1 = parsed['eq_id_1']
        eq_id_2 = parsed['eq_id_2']
        rxn_name = f"{network_name}-TS{ts_id_num} CON({eq_id_1},{eq_id_2})"
        ref_instance_index = None
        for n in range(len(ref_pkl["transition_state"]["rxn"])):
            if ref_pkl["transition_state"]["rxn"][n] == rxn_name:
                ref_instance_index = n
                break
        if ref_instance_index is not None:
            log_parsing_results = parse_grrm_saddlepoint_irc_log(log_files_dir/Path(log_file), num_atoms = num_atoms)
            ts_matching_results = compare_ref_vs_log_tss(log_parsing_results, ref_pkl, ref_instance_index, threshold = 12*threshold, num_to_symbol = num_to_symbol, mode = ts_cand_mode)
            eq_matching_results = compare_ref_vs_log_eqs(log_parsing_results, ref_pkl, ref_instance_index, threshold = threshold, num_to_symbol = num_to_symbol)
        else:
            print(f"{rxn_name}: reference index not found in the dataset.")
            continue
        temp_instance_results["initial_TS_has_single_negative_eigenvalue"] = log_parsing_results["initial_TS_single_neg_freq_confirmed"]
        temp_instance_results["TS_was_successfully_reoptimized"] = log_parsing_results["saddle_point_found"]

        # What is the best RMSD, if any, between re-optimized TS and reference TS?
        target_keys = [f"RMSD_of_{ts_cand_mode}_TS_to_ref_TS", f"RMSD_of_matched_mirrored_{ts_cand_mode}_TS_to_ref_TS"]
        valid_values = [ts_matching_results[k] for k in target_keys if k in ts_matching_results]
        temp_instance_results[f"best_{ts_cand_mode}_TS_match_RMSD"] = min(valid_values, default=None)

        # Are both IRC EQs matched with reference EQs? W
        temp_eq_matching = []
        temp_eq_rmsds = []
        for i in range(2): # IRC EQ index
            temp_eq_matching.append([])
            temp_eq_rmsds.append([])
            for j in range(2): # Ref EQ index
                if eq_matching_results[f"matched_irc_eq{i+1}_to_ref_eq{j+1}"]:
                    temp_eq_matching[i].append(j)
                    temp_eq_rmsds[i].append(eq_matching_results[f"RMSD_of_matched_irc_eq{i+1}_to_ref_eq{j+1}"])
                if eq_matching_results[f"matched_mirrored_irc_eq{i+1}_to_ref_eq{j+1}"]:
                    temp_eq_matching[i].append(j)
                    temp_eq_rmsds[i].append(eq_matching_results[f"RMSD_of_matched_mirrored_irc_eq{i+1}_to_ref_eq{j+1}"])
            temp_eq_rmsds[i] = [x for x in temp_eq_rmsds[i] if x is not None]

        temp_eq_rmsds = [item for sublist in temp_eq_rmsds for item in sublist]
        irc_eq1_set = set(temp_eq_matching[0])
        irc_eq2_set = set(temp_eq_matching[1])
        temp_instance_results["both_IRC_EQs_matched_ref_EQs"] = bool(irc_eq1_set) and bool(irc_eq2_set) and (bool(irc_eq1_set - irc_eq2_set) or bool(irc_eq2_set - irc_eq1_set))
        temp_instance_results["best_IRC_EQ_match_RMSDs"] = copy.copy(temp_eq_rmsds)

        # How different is the potential energy of the initial generated TS from the re-optimized TS?
        init_e = log_parsing_results.get("initial_TS_energy")
        reopt_e = log_parsing_results.get("reopt_TS_energy")
        if init_e is not None and reopt_e is not None:
            temp_instance_results["TS_reopt_Delta_energy"] = init_e - reopt_e
        else:
            temp_instance_results["TS_reopt_Delta_energy"] = None
        batch_log_results[rxn_name] = copy.deepcopy(temp_instance_results)
    return batch_log_results

#batch_log_results = batch_process_grrm_saddlepoint_irc_log_files(log_files_dir, ref_pkl_path, ts_cand_mode="reopt")
batch_log_results = batch_process_grrm_saddlepoint_irc_log_files(log_files_dir, ref_pkl_path)

Found optimized structure and energy.
Saddle point was found after 20 iterations.
Found optimized structure and energy.
Found optimized structure and energy.
Reference instance 5644: Acceptable RMSD was not found.
Reference instance 5644: Acceptable RMSD was not found.
Found optimized structure and energy.
Saddle point was found after 7 iterations.
Found optimized structure and energy.
Found optimized structure and energy.
Reference instance 3051: Acceptable RMSD was not found.
Reference instance 3051: Acceptable RMSD was not found.
Found optimized structure and energy.
Saddle point was found after 6 iterations.
Found optimized structure and energy.
Found optimized structure and energy.
Reference instance 5600: Acceptable RMSD was not found.
Reference instance 5600: Acceptable RMSD was not found.
Reference instance 5600: Acceptable RMSD was not found.
Found optimized structure and energy.
Saddle point was found after 13 iterations.
Found optimized structure and energy.
Found optimized 

In [46]:
initial_ts_has_negfreq = []
for i, (rxn_name, val) in enumerate(batch_log_results.items()):
    #print(i)
    initial_ts_has_negfreq.append(val["initial_TS_has_single_negative_eigenvalue"])
print(len([x for x in initial_ts_has_negfreq if x])/len(initial_ts_has_negfreq))

0.51


In [47]:
ts_was_optimized = []
for i, (rxn_name, val) in enumerate(batch_log_results.items()):
    #print(i)
    ts_was_optimized.append(val["TS_was_successfully_reoptimized"])
print(len([x for x in ts_was_optimized if x])/len(ts_was_optimized))

0.98


In [48]:
both_IRC_EQs_matched_ref_EQs = []
for i, (rxn_name, val) in enumerate(batch_log_results.items()):
    if val["TS_was_successfully_reoptimized"]:
        both_IRC_EQs_matched_ref_EQs.append(val["both_IRC_EQs_matched_ref_EQs"])
print(len([x for x in both_IRC_EQs_matched_ref_EQs if x])/len(both_IRC_EQs_matched_ref_EQs))

0.14285714285714285


In [49]:
len(both_IRC_EQs_matched_ref_EQs)

98

In [50]:
len([x for x in both_IRC_EQs_matched_ref_EQs if x])

14

In [51]:
# TS_reopt_Delta_energy
TS_reopt_Delta_energy = []
for i, (rxn_name, val) in enumerate(batch_log_results.items()):
    if val["TS_was_successfully_reoptimized"]:
        TS_reopt_Delta_energy.append(val["TS_reopt_Delta_energy"])
print(len([x for x in TS_reopt_Delta_energy if x is not None])/len(TS_reopt_Delta_energy))
print(np.mean(TS_reopt_Delta_energy))
print(np.std(TS_reopt_Delta_energy))

1.0
0.0037547641911633684
0.02164186282268856


In [52]:
ts_cand_mode = "initial"
best_cand_TS_match_RMSD = []
for i, (rxn_name, val) in enumerate(batch_log_results.items()):
    if val["TS_was_successfully_reoptimized"]:
        best_cand_TS_match_RMSD.append(val[f"best_{ts_cand_mode}_TS_match_RMSD"])
        #print(val["best_reopt_TS_match_RMSD"])
print(len([x for x in best_cand_TS_match_RMSD if x is not None])/len(best_cand_TS_match_RMSD))
if len([x for x in best_cand_TS_match_RMSD if x is not None]) != 0:
    print(np.mean([x for x in best_cand_TS_match_RMSD if x is not None]))
    print(np.std([x for x in best_cand_TS_match_RMSD if x is not None]))

0.4897959183673469
0.30661779933191946
0.16498718571213497


In [53]:
len([x for x in best_cand_TS_match_RMSD if x is not None])

48

In [57]:
best_IRC_EQ_match_RMSDs = []
for i, (rxn_name, val) in enumerate(batch_log_results.items()):
    if val["TS_was_successfully_reoptimized"]:
        for x in val["best_IRC_EQ_match_RMSDs"]:
            if x is not None:
                best_IRC_EQ_match_RMSDs.append(x)
            else:
                print(i)
        #print(val["best_reopt_TS_match_RMSD"])
print(len(best_IRC_EQ_match_RMSDs))
print(np.mean(best_IRC_EQ_match_RMSDs ))
print(np.std(best_IRC_EQ_match_RMSDs ))

81
0.0007931732186970618
0.0010716696132798051


In [58]:
best_IRC_EQ_match_RMSDs = []
for i, (rxn_name, val) in enumerate(batch_log_results.items()):
    if val["TS_was_successfully_reoptimized"]:
            if len([x for x in val["best_IRC_EQ_match_RMSDs"] if x is not None]) > 0:
                best_IRC_EQ_match_RMSDs.append(val["best_IRC_EQ_match_RMSDs"])
            else:
                pass
                #print(i)
        #print(val["best_reopt_TS_match_RMSD"])
print(len(best_IRC_EQ_match_RMSDs))
#print(np.mean(best_IRC_EQ_match_RMSDs))
#print(np.std(best_IRC_EQ_match_RMSDs))

57


In [59]:
57/98

0.5816326530612245

In [55]:
empty_count = 0

for i, (rxn_name, val) in enumerate(batch_log_results.items()):
    if val["TS_was_successfully_reoptimized"]:
        if len(val["best_IRC_EQ_match_RMSDs"]) == 0:
            empty_count += 1

In [56]:
empty_count

41